Neuron subclustering. Here, neurons (and undefined cells) are re-integrated, re-clustered and annotated with marker genes (WT, ECKO). Output: `…-reannotated.h5ad.gz` (input for 03, 04).

In [ ]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.pyplot import rc_context

sys.path.insert(1, "/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark")
import cellseg_benchmark._constants as markers
from cellseg_benchmark._constants import cell_type_colors
from cellseg_benchmark.adata_utils import clean_pca_umap, integration_harmony, normalize_counts, pca_umap, plot_spatial_multiplot

plt.rcParams["font.family"] = "Arial" if "Arial" in [f.name for f in fm.fontManager.ttflist] else "sans-serif"

In [ ]:
base_path = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark")
method_path = base_path / "analysis/foxf2/Cellpose_1_nuclei_model"
output_path = method_path / "neuronal-subclustering"
output_path.mkdir(exist_ok=True)

## Load

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/adata_integrated.h5ad.gz")
adata.obs["cell_type"] = pd.Categorical(
    adata.obs["cell_type_revised"].astype(str).replace({"Astroependymal": "Astrocytes"}), categories=list(cell_type_colors)
).remove_unused_categories()
adata.uns["cell_type_colors"] = [cell_type_colors[c] for c in adata.obs["cell_type"].cat.categories]
adata.obs["brain_region"] = adata.obs_names.map(pd.read_csv(method_path / "spatial_registration.csv", index_col=0)["label"])
with rc_context({"figure.figsize": (8, 8)}):
    sc.pl.embedding(adata, basis="X_umap_harmony_20_50", color="cell_type", size=220000 / adata.n_obs)

In [ ]:
region_colors = {
    "CTX": "#2986cc", "Fibre tract": "#f49c0c", "HIP": "#469e20", "DG-sg": "#1c4609", "CAsp": "#347b15",
    "BS": "#b0e92e", "BS/STR": "#cc0000", "STR": "#a569c7", "Meninges": "#bcbcbc", "VS": "#1acadc",
}
for ext in ("png", "svg"):
    plot_spatial_multiplot(adata, "brain_region", size=1.5, figsize_per_ax=5, palette=region_colors, title_keys=["sample", "genotype"],
                           save_path=output_path, save_name=f"spatial_brain_regions.{ext}")

In [ ]:
neurons = ["Neurons-Glut", "Neurons-Gaba", "Neurons-Dopa", "Neurons-Granule-Immature", "Neurons-Glyc-Gaba", "Neurons-Other", "Undefined"]
sub = adata[adata.obs["cell_type"].isin(neurons)].copy()
with rc_context({"figure.figsize": (8, 8)}):
    sc.pl.embedding(sub, basis="X_umap_harmony_20_50", color="cell_type", size=440000 / adata.n_obs)
del adata
for c in ["condition", "sample", "cell_type"]:
    display(sub.obs[c].value_counts())

## Re-integrate

In [ ]:
sub = clean_pca_umap(sub)
sub.X = sub.layers["counts"]
sub = normalize_counts(sub, seg_method="Cellpose_1_nuclei_model", save_path=None, trim_outliers=False)
assert round(sub.layers["volume_norm"].sum(1).mean()) == 250
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, layer in zip(axes, ["counts", "volume_log1p_norm", "zscore", "librarysize_log1p_norm"]):
    sns.histplot(sub.layers[layer].sum(1), bins=100, ax=ax, edgecolor="none", legend=False)
    ax.set_title(layer)
plt.tight_layout()
plt.show()

In [ ]:
sub.X = sub.layers["zscore"]
sub = pca_umap(sub, save_path=None)
for basis in ["X_umap_10_20", "X_umap_15_40", "X_umap_20_50"]:
    with rc_context({"figure.figsize": (8, 8)}):
        sc.pl.embedding(sub, basis=basis, color=["sample", "slide", "condition", "cell_type"], size=320000 / sub.n_obs, wspace=0.2)

In [ ]:
sub = integration_harmony(sub, batch_key="slide", save_path=None)
with rc_context({"figure.figsize": (8, 8)}):
    sc.pl.embedding(sub, basis="X_umap_harmony_20_50", color=["sample", "slide", "condition", "cell_type"], size=320000 / sub.n_obs, wspace=0.2)
sub.write(method_path / f"adatas/{date.today():%Y%m%d}_temp-after-neuronal-subclustering.h5ad.gz", compression="gzip")

## Marker scores (WT, ECKO)

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/20251114_temp-after-neuronal-subclustering.h5ad.gz")
adata = adata[adata.obs["genotype"].isin(["WT", "ECKO"])].copy()
adata.obsm["X_umap"] = adata.obsm["X_umap_harmony_20_50"].copy()
size = 250000 / adata.n_obs
with rc_context({"figure.figsize": (8, 8)}):
    sc.pl.embedding(adata, basis="X_umap", color=["sample", "slide", "genotype", "cell_type"], size=size, wspace=0.2)

In [ ]:
abc = pd.read_csv(base_path / "misc/scRNAseq_ref_ABCAtlas_Yao2023Nature/marker_genes_df/20250416_cell_type_markers_top50.csv")
marker_sets = {  # name: (markers, vmax of score UMAPs)
    "ABCAtlas": ({ct: abc[ct].dropna().tolist() for ct in abc.columns if ct not in ("0", "Bergmann")}, 0.99),
    "cell_cycle": (markers.cell_cycle_genes, 0.99),
    "cell_supertypes": (markers.cell_supertypes_merfish_panel_500_mouse_brain, 0.5),
    "cell_types": (markers.cell_types_merfish_panel_500_mouse_brain, 0.9),
    "EC_subtypes": (markers.vascular_EC_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
    "other_vascular_subtypes": (markers.vascular_nonEC_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
    "oligo_subtypes": (markers.oligo_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
    "astrocyte_subtypes": (markers.astro_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
    "neuronal_subtypes": (markers.neuronal_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
    "immune_subtypes": (markers.immune_cell_subtypes_merfish_panel_500_mouse_brain, 0.9),
}
plot_dir = method_path / f"plots-marker-genes/{date.today():%Y%m%d}_neuronal_subclusters"
scored = adata.copy()
for name, (sets, vmax) in marker_sets.items():
    sets = {ct: [g for g in genes if g in scored.var_names] for ct, genes in sets.items()}
    sets = {ct: genes for ct, genes in sets.items() if genes}
    for ct, genes in sets.items():
        sc.tl.score_genes(scored, genes, score_name=ct)
    plot_dir.mkdir(parents=True, exist_ok=True)
    with rc_context({"figure.figsize": (6, 6)}):
        sc.pl.umap(scored, color=list(sets), frameon=False, ncols=3, cmap="RdYlBu_r", vmin=0, vmax=vmax, size=size, sort_order=True, show=False)
    plt.savefig(plot_dir / f"{name}_UMAP_scores.png", bbox_inches="tight")
    plt.close()
    if name in ("cell_supertypes", "cell_types"):
        continue
    (plot_dir / f"{name}_UMAP_genes").mkdir(exist_ok=True)
    for ct, genes in sets.items():
        sc.pl.umap(scored, color=genes, frameon=False, ncols=3, vmin=0, vmax="p99", sort_order=True, cmap="Reds", show=False)
        plt.savefig(plot_dir / f"{name}_UMAP_genes/{ct}.png", bbox_inches="tight")
        plt.close()
del scored

## Clustering

In [ ]:
adata.uns["neighbors"] = adata.uns["neighbors_harmony_20_50"]
adata.obsp["connectivities"] = adata.obsp["neighbors_harmony_20_50_connectivities"]
adata.obsp["distances"] = adata.obsp["neighbors_harmony_20_50_distances"]
resolutions = [0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 1.5, 2.0]
for r in resolutions:
    sc.tl.leiden(adata, key_added=f"leiden_res{r}".replace(".", "_"), resolution=r, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=[f"leiden_res{r}".replace(".", "_") for r in resolutions], legend_loc="on data", s=0.5, ncols=4,
           legend_fontoutline=2, legend_fontsize=12, legend_fontweight="normal")

In [ ]:
leiden = "leiden_res0_25"
adata = adata[adata.obs.groupby(leiden, observed=True)[leiden].transform("size") > 1].copy()  # drop single-cell clusters
with rc_context({"figure.figsize": (10, 10)}):
    sc.pl.umap(adata, color=leiden, legend_loc="on data", s=size, legend_fontoutline=4, legend_fontsize=12, legend_fontweight="normal")
sc.tl.rank_genes_groups(adata, groupby=leiden, method="wilcoxon", key_added="dea_leiden")
sc.tl.dendrogram(adata, groupby=leiden)
sc.pl.rank_genes_groups_dotplot(adata, groupby=leiden, standard_scale="var", n_genes=4, key="dea_leiden")
pd.DataFrame(adata.uns["dea_leiden"]["names"]).head(10)

## Annotation

In [ ]:
old_to_new = {
    "0": "Neurons-Gaba",
    "1": "Neurons-Glut",
    "2": "Neurons-Gaba",
    "3": "Neurons-Mixed",  # neuronal + immune
    "4": "Neurons-Glut",
    "5": "Neurons-Glut",
    "6": "Neurons-Gaba",
    "7": "Neurons-Gaba",
    "8": "Neurons-Gaba",
    "9": "Neurons-Glut",
    "10": "Neurons-Other",
    "11": "Neurons-Mixed",  # neuronal + oligodendrocyte
    "12": "Neurons-Glut",
    "13": "Neurons-Glut",
    "14": "Neurons-Glut",
}
adata.obs["cell_type_sub"] = pd.Categorical(adata.obs[leiden].map(old_to_new), categories=["Neurons-Glut", "Neurons-Gaba", "Neurons-Other", "Neurons-Mixed"])
palette = {**cell_type_colors, "Neurons-Mixed": "#D9D9D9", "Neurons-Glut-lowNeuN": "red"}
with plt.style.context("default"), rc_context({"figure.figsize": (8, 8)}):
    sc.pl.umap(adata, color="cell_type_sub", palette=palette, size=2, legend_fontoutline=2, legend_fontsize=13, legend_fontweight="normal")
plot_spatial_multiplot(adata, "cell_type_sub", size=0.4, figsize_per_ax=2.5)
adata.write(method_path / f"adatas/{date.today():%Y%m%d}_temp-after-neuronal-subclustering-reannotated.h5ad.gz", compression="gzip")

## Cortex labels on the full object (plots only)

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/20251111_temp-after-neuronal-subclustering-reannotated.h5ad.gz")
sc.pl.dotplot(adata, ["Rbfox3"], groupby="cell_type_sub")
adata = adata[adata.obs["brain_region"] == "CTX"].copy()
plot_spatial_multiplot(adata, "brain_region", size=0.4, figsize_per_ax=2.5)

In [ ]:
full = sc.read_h5ad(method_path / "adatas/adata_integrated.h5ad.gz")
full.obs["brain_region"] = full.obs_names.map(pd.read_csv(method_path / "spatial_registration.csv", index_col=0)["label"])
ct = full.obs["cell_type_revised"].astype(str).replace({"Astroependymal": "Astrocytes"})
sub = adata.obs["cell_type_sub"].reindex(full.obs_names).astype(object)
labels = sub.mask(ct.isin(neurons) & sub.isna(), "Excluded").fillna(ct)  # neurons outside the subclustered set: Excluded
colors = {**palette, "Excluded": "#D3D3D3"}
full.obs["cell_type_sub"] = pd.Categorical(labels, categories=[c for c in colors if c in set(labels)])
full.uns["cell_type_sub_colors"] = [colors[c] for c in full.obs["cell_type_sub"].cat.categories]
plot_spatial_multiplot(full, "cell_type_sub", size=4.5, figsize_per_ax=7.5, title_keys=["sample", "genotype"])
plot_spatial_multiplot(full[full.obs["brain_region"] == "CTX"].copy(), "cell_type_sub", size=4.5, figsize_per_ax=7.5, title_keys=["sample", "genotype"])
with rc_context({"figure.figsize": (8, 8)}):
    sc.pl.embedding(full, basis="X_umap_harmony_20_50", color="cell_type_sub", size=500000 / full.n_obs)